<a href="https://colab.research.google.com/github/johnpaix-ez1/newvidlands/blob/feat%2Fcolab-conversion/video_pipeline_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Video Pipeline on Google Colab - Setup

## 1. Install Dependencies

In [28]:
!pip install -q google-generativeai jsonschema yt-dlp moviepy numpy groq websocket-client Pillow openai-whisper requests pyspellchecker python-dotenv termcolor kokoro-onnx onnxruntime soundfile

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00


## 1a. Clone Repository (Optional but Recommended for Assets)

If you haven't manually uploaded the `assets` folder from the repository to your Google Drive, you can clone the repository here. This will download the project structure, including the `assets` directory (with default fonts, example workflows, etc.) into the Colab temporary environment.

**Important:** You'll still need to ensure that essential large files (like Kokoro TTS models) and your primary assets are placed in the Google Drive `ASSETS_DIR` defined in Cell 4, as Colab's local storage is temporary. You can use the file browser on the left to copy files from the cloned repo (e.g., `/content/YOUR_REPOSITORY_NAME/assets/`) to your Google Drive `ASSETS_DIR`.

In [29]:
# IMPORTANT: Replace the URL with your actual repository URL if it's different!
!git clone https://github.com/johnpaix-ez1/newvidlands.git
%cd newvidlands
# You can then navigate the cloned repository files using the file browser on the left (usually under /content/YOUR_REPOSITORY_NAME)


Cloning into 'newvidlands'...
remote: Enumerating objects: 151, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 151 (delta 63), reused 109 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (151/151), 142.43 KiB | 7.91 MiB/s, done.
Resolving deltas: 100% (63/63), done.
/content/newvidlands/newvidlands


In [30]:
from google.colab import drive
drive.mount('/content/drive')
!pwd
# For Google Colab, you would typically use: from google.colab import userdata
# GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
# GROQ_API_KEY = userdata.get('GROQ_API_KEY')

GEMINI_API_KEY = input('Enter your Google Gemini API Key: ')
GROQ_API_KEY = input('Enter your Groq API Key: ')

# Step 1: Change to your desired directory (make sure it exists)
%cd /content/drive/MyDrive/VideoPipelineProject/assets

# Step 2: Download the Kokoro ONNX model and voices file using wget
!wget https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx
!wget https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin

# Step 3: (Optional) List files to confirm download
!ls -lh


/content/newvidlands/newvidlands


In [36]:
!pwd

/content/drive/MyDrive/VideoPipelineProject/assets


## 4. Define File Paths

Adjust `DRIVE_PROJECT_BASE_PATH` if you place this project in a subdirectory within your Google Drive.

## Main Script Logic

In [60]:
from IPython import get_ipython
from IPython.display import display
import os
import sys
import json
import pathlib
import logging
import asyncio
import time
import re
import uuid
import urllib.parse
import io
import subprocess
import shutil
import random
import math

# Base path for the project in your Google Drive
DRIVE_PROJECT_BASE_PATH = pathlib.Path('/content/drive/MyDrive/VideoPipelineProject') # CHANGE THIS IF NEEDED

INPUT_DIR = DRIVE_PROJECT_BASE_PATH / 'input'
WORKSPACE_DIR = DRIVE_PROJECT_BASE_PATH / 'workspace'
FINAL_VIDEO_DIR = DRIVE_PROJECT_BASE_PATH / 'final_videos'
LOG_DIR = DRIVE_PROJECT_BASE_PATH / 'logs'
ASSETS_DIR = DRIVE_PROJECT_BASE_PATH / 'assets'

# Ensure base directories exist (optional, script also creates them)
pathlib.Path(INPUT_DIR).mkdir(parents=True, exist_ok=True)
pathlib.Path(WORKSPACE_DIR).mkdir(parents=True, exist_ok=True)
pathlib.Path(FINAL_VIDEO_DIR).mkdir(parents=True, exist_ok=True)
pathlib.Path(LOG_DIR).mkdir(parents=True, exist_ok=True)
pathlib.Path(ASSETS_DIR).mkdir(parents=True, exist_ok=True)

print(f'Input directory: {INPUT_DIR}')
print(f'Workspace directory: {WORKSPACE_DIR}')
print(f'Assets directory: {ASSETS_DIR}')
#==================================================================================================================================
# --- ComfyUI Configuration ---
COMFYUI_SERVER_ADDRESS = '127.0.0.1:8188' # Replace with your ComfyUI server address if not local or using a tunnel
COMFYUI_WORKFLOW_FILE_NAME = 'default_comfyui_workflow.json' # Name of the workflow file in ASSETS_DIR
COMFYUI_WORKFLOW_FILE = ASSETS_DIR / COMFYUI_WORKFLOW_FILE_NAME
print(f'ComfyUI Server Address: {COMFYUI_SERVER_ADDRESS}')
print(f'ComfyUI Workflow File: {COMFYUI_WORKFLOW_FILE}')

# --- Kokoro TTS Configuration ---
# IMPORTANT: You MUST upload these model files to your Google Drive and update the paths here.
KOKORO_MODEL_FILE_PATH = ASSETS_DIR / 'kokoro_model.onnx'  # EXAMPLE: Update this path
KOKORO_VOICES_FILE_PATH = ASSETS_DIR / 'kokoro_voices.json' # EXAMPLE: Update this path
print(f'Kokoro Model Path: {KOKORO_MODEL_FILE_PATH}')
print(f'Kokoro Voices Path: {KOKORO_VOICES_FILE_PATH}')

# --- Other Asset Paths ---
ENDSCREEN_VIDEO_FILE_NAME = 'default_endscreen.mp4' # Name of the endscreen video in ASSETS_DIR
ENDSCREEN_VIDEO_FILE = ASSETS_DIR / ENDSCREEN_VIDEO_FILE_NAME
print(f'Endscreen Video File: {ENDSCREEN_VIDEO_FILE}')

# Environment variables for the script (some might be overridden by Colab inputs/config)
# GEMINI_API_KEY and GROQ_API_KEY are typically set via Colab secrets or user input cells earlier
# For this combined block, you might need to define them directly if not set globally beforehand:
# GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY') # Assume set by earlier cells
# GROQ_API_KEY = os.environ.get('GROQ_API_KEY')   # Assume set by earlier cells

os.environ['COMFYUI_SERVER_ADDRESS'] = COMFYUI_SERVER_ADDRESS
os.environ['COMFYUI_WORKFLOW_FILE'] = str(COMFYUI_WORKFLOW_FILE)
os.environ['KOKORO_MODEL_FILE_PATH'] = str(KOKORO_MODEL_FILE_PATH)
os.environ['KOKORO_VOICES_FILE_PATH'] = str(KOKORO_VOICES_FILE_PATH)
os.environ['ENDSCREEN_VIDEO_FILE'] = str(ENDSCREEN_VIDEO_FILE)
# YTDLP_COOKIES_FILE is not explicitly handled here by user input,
# but if the user uploads a cookies file to ASSETS_DIR (e.g., 'cookies.txt'),
# they can set YTDLP_COOKIES_FILE = str(ASSETS_DIR / 'cookies.txt') in a new cell before running the main script.
os.environ['YTDLP_COOKIES_FILE'] = '' # Default to empty if not set by user later


# --- Attempt to import third-party libraries, with fallbacks ---
try:
    import google.generativeai as genai
    from google.generativeai.types import HarmCategory, HarmBlockThreshold
except ImportError:
    genai = None
    HarmCategory = None
    HarmBlockThreshold = None
    print("WARNING: Gemini library (google.generativeai) not found. Script generation (Step 03) will use placeholders or be skipped.")

try:
    import jsonschema
except ImportError:
    jsonschema = None
    print("WARNING: jsonschema library not found. JSON validation (e.g., in Step 03) will be skipped.")

try:
    from kokoro_onnx import Kokoro
    import soundfile as sf
    KOKORO_AVAILABLE = True
    print("INFO: Kokoro-ONNX and soundfile imported successfully for TTS.")
except ImportError:
    KOKORO_AVAILABLE = False
    Kokoro = None # Define Kokoro as None if import fails
    sf = None
    print("WARNING: kokoro_onnx or soundfile library not found. Kokoro TTS (Step 04) will use a dummy WAV file.")

try:
    import whisper # from openai_whisper
except ImportError:
    whisper = None
    print("WARNING: OpenAI Whisper library not found. Transcription (Step 05) will use a dummy transcript.")

try:
    from spellchecker import SpellChecker
except ImportError:
    SpellChecker = None
    print("WARNING: pyspellchecker library not found. Spell correction (Step 06) will be skipped.")

try:
    from groq import Groq
except ImportError:
    Groq = None
    print("WARNING: Groq library not found. Image prompt generation (Step 08) will use dummy prompts.")

try:
    import websocket # For ComfyUI
    from PIL import Image, ImageDraw, ImageFont
except ImportError:
    websocket = None
    Image = None
    ImageDraw = None
    ImageFont = None
    print("WARNING: websocket-client or Pillow not found. ComfyUI image generation (Step 09) will use dummy images.")

try:
    import yt_dlp
except ImportError:
    yt_dlp = None
    print("WARNING: yt-dlp library not found. Video link processing (Step 02) will fail if attempted.")

moviepy_ok = False
ImageClip_cls, CompositeVideoClip_cls, VideoFileClip_cls, AudioFileClip_cls = None, None, None, None
TextClip_cls, ColorClip_cls = None, None
concatenate_videoclips_func, concatenate_audioclips_func, CompositeAudioClip_cls = None, None, None
moviepy_vfx_all = None
try:
    from moviepy.editor import (
        VideoFileClip, ImageClip, AudioFileClip, concatenate_videoclips,
        TextClip, CompositeVideoClip, concatenate_audioclips, ColorClip
    )
    import moviepy.video.fx.all as vfx
    from moviepy.audio.AudioClip import CompositeAudioClip
    ImageClip_cls = ImageClip
    CompositeVideoClip_cls = CompositeVideoClip
    VideoFileClip_cls = VideoFileClip
    AudioFileClip_cls = AudioFileClip
    TextClip_cls = TextClip
    ColorClip_cls = ColorClip
    concatenate_videoclips_func = concatenate_videoclips
    concatenate_audioclips_func = concatenate_audioclips
    CompositeAudioClip_cls = CompositeAudioClip
    moviepy_vfx_all = vfx
    moviepy_ok = True
    print("INFO: MoviePy library and components loaded successfully.")
except ImportError:
    print("WARNING: MoviePy library or its components not found. Video processing steps (animation, assembly, captioning) will use fallbacks or fail.")

try:
    from termcolor import cprint
except ImportError:
    cprint = lambda text, color=None, on_color=None, attrs=None: print(text)
    print("WARNING: termcolor library not found. Console output will not be colored.")

# Basic logging setup
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


# API Keys & Configuration from environment (set in Colab setup cells)
YTDLP_COOKIES_FILE = os.getenv("YTDLP_COOKIES_FILE")
COMFYUI_SERVER_ADDRESS = os.getenv("COMFYUI_SERVER_ADDRESS", "127.0.0.1:8188")
COMFYUI_WORKFLOW_FILE = os.getenv("COMFYUI_WORKFLOW_FILE", str(ASSETS_DIR / "default_comfyui_workflow.json"))
ENDSCREEN_VIDEO_FILE = os.getenv("ENDSCREEN_VIDEO_FILE", str(ASSETS_DIR / "default_endscreen.mp4"))
KOKORO_MODEL_FILE_PATH = os.getenv("KOKORO_MODEL_FILE_PATH")
KOKORO_VOICES_FILE_PATH = os.getenv("KOKORO_VOICES_FILE_PATH")

# Ensure ASSETS_DIR subdirectories exist (fonts, bgsound)
# These are referenced by the script, good to ensure they are created if not by user.
if ASSETS_DIR:
    pathlib.Path(ASSETS_DIR / "fonts").mkdir(parents=True, exist_ok=True)
    pathlib.Path(ASSETS_DIR / "bgsound").mkdir(parents=True, exist_ok=True)


def ensure_dir_exists(path: pathlib.Path):
    """Creates a directory if it doesn't exist."""
    path.mkdir(parents=True, exist_ok=True)

def get_source_id(source_path: str) -> str:
    """Creates a unique ID for each input source."""
    # Simplified to use the filename stem for text inputs
    source_name = pathlib.Path(source_path).stem
    return f"{source_name}_{uuid.uuid4().hex[:8]}"

def get_workspace_path(source_id: str) -> pathlib.Path:
    """Gets the dedicated workspace for a source."""
    # WORKSPACE_DIR is globally defined from Colab setup
    return WORKSPACE_DIR / source_id

def is_step_complete(workspace_path: pathlib.Path, artifact_name: str) -> bool:
    """Checks if the .complete marker file for an artifact exists."""
    marker_file = workspace_path / f"{artifact_name}.complete"
    return marker_file.exists()

def mark_step_complete(workspace_path: pathlib.Path, artifact_name: str):
    """Creates a .complete marker file for an artifact."""
    ensure_dir_exists(workspace_path)
    (workspace_path / f"{artifact_name}.complete").touch()
    logger.info(f"Marked step as complete by creating: {workspace_path / artifact_name}.complete")

def load_json_config(file_path: pathlib.Path) -> dict | list | None:
    """Loads a JSON file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except FileNotFoundError:
        logger.error(f"JSON file not found: {file_path}")
        return None
    except json.JSONDecodeError:
        logger.error(f"Error decoding JSON from file: {file_path}")
        return None

def save_json_output(file_path: pathlib.Path, data):
    """Saves data to a JSON file."""
    ensure_dir_exists(file_path.parent)
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4)
    logger.info(f"Saved JSON output to: {file_path}")

def normalize_text(text: str) -> str:
    """Basic text normalization: remove asterisks, multiple spaces."""
    text = text.replace("*", "")
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# --- Step 1 Definition (Modified) ---
def step_01_process_text_input(input_text_path_str: str, workspace_path: pathlib.Path) -> str | None:
    logger.info(f"Called step_01_process_text_input with input: {input_text_path_str}")
    time.sleep(1)  # Simulate processing delay

    output_artifact_name = "processed_text.json"
    output_path = workspace_path / output_artifact_name
    ensure_dir_exists(output_path.parent)

    processed_content = "" # Initialize processed_content
    status = "error" # Initialize status to error

    try:
        # Check if the input path exists before trying to read
        if not pathlib.Path(input_text_path_str).exists():
             logger.error(f"Input file does not exist at {input_text_path_str} for step_01.")
             status = "error_file_not_found"
        else:
            # Try to load the input as JSON (if it's a .json file)
            if input_text_path_str.lower().endswith(".json"):
                with open(input_text_path_str, "r", encoding="utf-8") as f:
                    input_data = json.load(f)

                # === MODIFICATION START ===
                # If the input is a list of strings, take the first one
                if isinstance(input_data, list) and len(input_data) > 0 and isinstance(input_data[0], str):
                    processed_content = input_data[0] # Take only the first string
                    status = "success" # Assume success if we got at least one string
                    logger.info(f"Input JSON is a list of strings, taking the first item.")
                # If the JSON is a dict with a 'text' or 'content' field, use that
                elif isinstance(input_data, dict):
                    content_from_dict = input_data.get("text") or input_data.get("content")
                    if isinstance(content_from_dict, str):
                        processed_content = content_from_dict
                        status = "success" if processed_content.strip() else "error_empty_content_from_dict"
                        logger.info(f"Input JSON is a dictionary, extracted content from 'text' or 'content' field.")
                    else:
                         # If the dict didn't have 'text'/'content' or they weren't strings, serialize the whole dict
                         processed_content = json.dumps(input_data, indent=4)
                         status = "success" # Treat serialization as success, though content might be complex
                         logger.warning("Input JSON is a dictionary but no string found in 'text'/'content', serializing whole dict.")

                # If it's a JSON object but not a list or expected dict format, serialize it
                else:
                    processed_content = json.dumps(input_data, indent=4)
                    status = "success" # Treat serialization as success
                    logger.warning(f"Input JSON is neither a list of strings nor a dict with 'text'/'content'. Serialized the entire object (type: {type(input_data)}).")

                # Final check for success status after processing JSON
                if status == "success" and not processed_content.strip():
                     status = "error_empty_content" # Correct status if content is empty after extraction/serialization


                # === MODIFICATION END ===

            else:
                # Otherwise, treat as plain text
                with open(input_text_path_str, "r", encoding="utf-8") as f:
                    processed_content = f.read()
                status = "success" if processed_content.strip() else "error_empty_content" # Check if content is not just whitespace


    except json.JSONDecodeError as e:
         logger.error(f"Failed to decode JSON from {input_text_path_str} for step_01: {e}")
         status = "error_json_decode"
         processed_content = "" # Ensure content is empty on decode error

    except Exception as e:
        logger.error(f"Failed to load input text for step_01: {e}")
        status = "error_read_failed"
        processed_content = "" # Ensure content is empty on other read errors

    # Save the output JSON regardless of status
    # processed_content will now be either the first string (if input was list of strings)
    # or the content from text/content key (if input was dict)
    # or the serialized JSON/plain text
    save_json_output(output_path, {
        "status": status,
        "input_received": input_text_path_str,
        "processed_content": processed_content # Save the extracted/processed content
    })

    # Only mark complete and return path if the step was successful in getting non-empty content
    if status == "success" and processed_content.strip():
        mark_step_complete(workspace_path, "processed_text")
        logger.info(f"step_01_process_text_input completed. Output: {str(output_path)}")
        return str(output_path) # Return the path only on success
    else:
        logger.error(f"Step 01 failed to produce usable content (status: {status}). Output JSON saved with status details: {str(output_path)}")
        return None # Indicate failure


# --- Other Step Definitions (truncated for brevity) ---
# ... (Include the definitions for step_02 through step_10 here)
# You should copy these from your existing code.
# I'll include placeholder definitions to make the main function runnable.



def step_02_process_video_link_input(video_url_str: str, workspace_path: pathlib.Path, cookies_file_path: str | None = None) -> str | None:
    """
    Downloads video info (and optionally the video) using yt-dlp for a given URL.
    If video_url_str is a path to a file (e.g., in 'input/link_sources'), loads the first non-empty line as the URL.
    """
    logger.info(f"Called step_02_process_video_link_input with URL or file: {video_url_str}")
    time.sleep(1)  # Simulate processing delay

    if not yt_dlp:
        logger.warning("yt-dlp library not found. Step 02 cannot process video links.")
        output_artifact_name = "downloaded_video_info.json"
        output_path = workspace_path / output_artifact_name
        ensure_dir_exists(output_path.parent)
        save_json_output(output_path, {"status": "dummy output from placeholder step_02", "video_url": video_url_str, "comment": "yt-dlp missing or actual download skipped"})
        mark_step_complete(workspace_path, "downloaded_video_info")
        logger.info(f"step_02_process_video_link_input completed. Output: {str(output_path)}")
        return str(output_path)

    # If input is a file, load the first non-empty line as the URL
    url = video_url_str
    # Check if video_url_str is a path and if it exists
    if pathlib.Path(video_url_str).is_file():
        try:
            with open(video_url_str, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line and not line.startswith("#"):
                        url = line
                        break
            logger.info(f"Loaded URL from file: {url}")
        except Exception as e:
            logger.error(f"Could not read URL from file {video_url_str}: {e}")
            # Fallback to using video_url_str as a direct URL if file reading fails
            url = video_url_str

    output_artifact_name = "downloaded_video_info.json"
    output_path = workspace_path / output_artifact_name
    ensure_dir_exists(output_path.parent)

    ydl_opts = {
        "skip_download": True,
        "quiet": True,
        "no_warnings": True,
        "ignoreerrors": True,
        "forcejson": True,
        "simulate": True,
    }
    # Use YTDLP_COOKIES_FILE from env if set, otherwise use cookies_file_path param
    effective_cookies_file = YTDLP_COOKIES_FILE or cookies_file_path
    if effective_cookies_file and pathlib.Path(effective_cookies_file).exists():
        ydl_opts["cookiefile"] = str(effective_cookies_file)
        logger.info(f"Using cookies file: {effective_cookies_file}")
    elif effective_cookies_file: # if path was provided but file doesn't exist
        logger.warning(f"Cookies file specified but not found: {effective_cookies_file}")

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=False)
        save_json_output(output_path, {"status": "success", "video_url": url, "video_info": info})
        mark_step_complete(workspace_path, "downloaded_video_info")
        logger.info(f"step_02_process_video_link_input completed. Output: {str(output_path)}")
        return str(output_path)
    except Exception as e:
        logger.error(f"yt-dlp failed to extract info for '{url}': {e}")
        save_json_output(output_path, {"status": "error", "video_url": url, "error": str(e)})
        # Still mark as complete to avoid re-running a failing download, but log error
        mark_step_complete(workspace_path, "downloaded_video_info")
        return str(output_path) # Return path to error JSON
GEMINI_SCRIPT_SCHEMA = {
    "type": "object",
    "properties": {
        "new_video_title": {"type": "string"},
        "script": {"type": "string"},
        "keywords": {"type": "array", "items": {"type": "string"}}
    },
    "required": ["new_video_title", "script", "keywords"]
}

def validate_json_structure(data, schema):
    """Validate JSON data against a schema."""
    if not jsonschema:
        logger.warning("jsonschema library not available. Skipping validation.")
        return True
    try:
        jsonschema.validate(instance=data, schema=schema)
        return True
    except jsonschema.ValidationError as e:
        logger.error(f"Validation error: {e}")
        return False

def step_03_generate_gemini_script(processed_text_path_str: str, workspace_path: pathlib.Path, script_instructions: str | None = None) -> str | None:
    logger.info(f"Called step_03_generate_gemini_script with text: {processed_text_path_str}")
    time.sleep(3)

    if not genai:
        logger.error("Gemini library (google.generativeai) not found. Step 03 cannot proceed.")
        return None
    if not os.getenv("GEMINI_API_KEY"):
        logger.error("GEMINI_API_KEY not found in environment. Step 03 cannot proceed.")
        return None
    genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

    try:
        with open(processed_text_path_str, "r", encoding="utf-8") as f:
            processed_data = json.load(f)
        transcript = processed_data.get("processed_content", "")
    except Exception as e:
        logger.error(f"Failed to load processed text for Gemini script generation: {e}")
        return None

    prompt_template = script_instructions if script_instructions else """You are an expert content creator for a YouTube channel that produces concise, engaging, and insightful videos. make it just 25 to 30 seconds (100 to 120 words) long. Your task is to analyze the provided transcript and transform it into an **exceptionally compelling and unforgettable YouTube script** that makes viewers feel they've stumbled upon something truly unique and insightful. The goal is to maximize deep engagement, watch time, and a desire to share.
    Your task is to analyze the provided transcript and transform it into an **exceptionally compelling and unforgettable YouTube script** that makes viewers feel they've stumbled upon something truly unique and insightful. The goal is to maximize deep engagement, watch time, and a desire to share.

    - **Craft an Electrifying Introduction:** Start with a hook that is not just attention-grabbing but *paradigm-shifting* for the viewer regarding the topic. This could be a startling reframe of a common assumption, a deeply counter-intuitive question, a bold, almost unbelievable claim (that the script will then substantiate), or a vivid, unexpected analogy related to the transcript's core message. Aim for immediate intrigue and a "I *need* to know more" reaction. Examples of powerful hook approaches (adapt to the content): 'What if everything you thought you knew about X was a carefully constructed illusion?', 'The one tiny detail about Y that changes absolutely everything...', 'Forget X, Y, and Z; the *real* story behind [topic] is far more [adjective] than anyone dares to admit.'

    - **Deliver Content as a Riveting Unveiling:** Present the information not just clearly, but as a journey of discovery. Employ dynamic, conversational language that feels like an insider sharing groundbreaking secrets.
        - Use leading phrases that build anticipation and a sense of unique insight: 'But here's where it gets truly mind-bending...', 'The hidden layer most people miss is...', 'Consider this unexpected connection...', 'What if the real story is far stranger/simpler/more profound than we're led to believe?'
        - Weave in moments of **dramatic emphasis, unexpected comparisons, or sharp contrasts** to make key points land with impact and stick in the viewer's mind.
        - The script must feel like a human sharing a passionate, almost urgent message, not a dry recitation of facts. Infuse it with genuine curiosity and a sense of wonder or revelation.

    - **Engineer Engagement:**
        - **Maximize Curiosity Gaps:** Strategically pose questions or hint at revelations that compel viewers to keep watching to find the answer.
        - **Introduce Unexpected Twists or Perspectives:** If the transcript allows, present information in a way that challenges common perceptions or reveals a surprising angle on the topic. This should be done without being misleading or resorting to clickbait.
        - **Amplify Emotional Resonance (Authentically):** Where appropriate to the content, connect with the viewer on an emotional level. This isn't about forced sentimentality, but about highlighting the human impact, the 'wow' factor, the profound implications, or the sheer fascination of the information. Use vivid, evocative language.
        - **Viral-Optimized Tone (Substance over Hype):** The tone should be compelling and shareable, like top-tier educational or insight-driven YouTube channels. Focus on making the *substance itself* feel shocking, surprising, or highly relevant, rather than just using superficial hype. Persuasive language should stem from the power of the ideas being presented.
        - **Narrative Drive:** Structure the script like an unfolding mystery, a compelling argument being built piece by piece, or a journey to an 'aha!' moment. Ensure each segment logically and excitingly leads to the next.

    - **Maintain Factual Integrity and Clarity:**
        - Use simple to understand english for the understanding of someone who english is not thier first language. Do not use terms thats that need a dictionary to get its meaning, rather simpler word used in day to day conversation which still perfectly delivers the message.
        - Don't mention the transcript even if its the source from which you got this information explain. Stop mentioning the source.
        - If its a news, report it as news not as a story clearly articulating what the news said
        - Avoids repeating exact words from the transcript by using synonyms and expanding with related ideas or examples.
        - Examine the content and determine whether it qualifies as news or should be categorized as opinion, analysis, or feature. Consider factors like timeliness, factual accuracy, and relevance in your judgment.
        - As you generate the script, cross-check any factual claims, dates, figures, or reported events with your own knowledge and understanding. If the content appears outdated, inaccurate, or misleading, adjust it accordingly to ensure factual accuracy and clarity. Do not include unverifiable or misleading claims in the final script.
        - If the video transcript, the author mentioned his or her name, focus only on the script and don't mention the persons name in the generated script.

    The script should also adhere to these specific formatting and style points:
    - Do not include asterisks (*) or emojis.
    - Begin with one of your example hooks (e.g.,'Hey ...', or 'Stop scrolling ...', 'What if I told you...', 'Here is some breaking news...'), ensuring it aligns with the 'Electrifying Introduction' goal above.
    - Flow seamlessly from one point to the next with transitional phrases. If the original script implies or contains numbered points, you can structure the generated script with numbering (e.g., first on our list, second, third...).
    - **Conclude with Impact and a Compelling Call to Action:** End not just with an intriguing question, but with one that **challenges the viewer's perspective or ignites a desire for further exploration/discussion.** For the call to action, instead of a generic "subscribe," frame it uniquely. For example: "If you're ready to keep uncovering the [adjective, e.g., 'hidden truths', 'extraordinary insights'] behind [channel's general topic], make sure you join our community of curious minds by subscribing and hitting that notification bell – you won't want to miss what's next."

    Additionally, generate:
    - A **new video title** that is catchy, informative, and optimized for YouTube.
    - A list of **keywords** relevant to the video's topic to enhance SEO and discoverability.

    Here's the provided transcript:

    {transcript}

    Please return the output as a JSON object in the following format:
    {{
        "new_video_title": "Your catchy video title",
        "keywords": ["keyword1", "keyword2", "keyword3"],
        "script": "The generated content here"
    }}
    """
    prompt = prompt_template.format(transcript=transcript)

    try:
        model = genai.GenerativeModel("gemini-1.5-flash") # Updated model name
        response = model.generate_content(prompt)
        logger.info("Raw Gemini API Response: %s", response.text)
        content = response.text

        json_match = re.search(r'```json\s*(.*?)\s*```', content, re.DOTALL)
        if json_match:
            json_str = json_match.group(1)
        else:
            # Fallback: if no ```json ``` block, try to find first '{' and last '}'
            first_brace = content.find('{')
            last_brace = content.rfind('}')
            if first_brace != -1 and last_brace != -1 and last_brace > first_brace:
                json_str = content[first_brace:last_brace+1]
            else:
                 logger.error(f"Could not extract JSON from Gemini response: {content}")
                 return None

        result = json.loads(json_str)

        if validate_json_structure(result, GEMINI_SCRIPT_SCHEMA):
            script_text = normalize_text(result["script"])
            script_text = script_text.replace("*", "")
            script_text = re.sub(r'\([^)]*\)', '', script_text) # Remove text in parentheses
            result["script"] = script_text

            output_artifact_name = "generated_script_gemini.json"
            output_path = workspace_path / output_artifact_name
            ensure_dir_exists(output_path.parent)
            save_json_output(output_path, {"status": "success", "input_processed_text": processed_text_path_str, "script": result})
            mark_step_complete(workspace_path, "generated_script_gemini")
            logger.info(f"step_03_generate_gemini_script completed. Output: {str(output_path)}")
            return str(output_path)
        else:
            logger.error("Gemini output JSON validation failed.")
            return None

    except json.JSONDecodeError as e:
        logger.error(f"JSON decoding error from Gemini response: {e}. Response content: {content}")
    except Exception as e:
        logger.error(f"Error during Gemini script generation: {e}")

    return None
def step_04_generate_tts_kokoro(script_path_str: str, workspace_path: pathlib.Path) -> str | None:
    logger.info(f"Called step_04_generate_tts_kokoro with script: {script_path_str}")
    time.sleep(1)

    output_artifact_name = "voiceover.wav"
    output_path = workspace_path / output_artifact_name
    ensure_dir_exists(output_path.parent)

    try:
        with open(script_path_str, "r", encoding="utf-8") as f:
            script_data = json.load(f)
        script_text = script_data.get("script", {}).get("script", "")
        if not script_text.strip():
            logger.error("Script text is empty or invalid in step 3 output.")
            return None
    except Exception as e:
        logger.error(f"Failed to load script for TTS: {e}")
        return None

    if not KOKORO_AVAILABLE or not Kokoro or not sf:
        logger.warning("Kokoro TTS or soundfile not available. Creating dummy WAV.")
        # Create a short, silent dummy WAV file
        try:
            import numpy as np
            dummy_audio = np.zeros(int(0.5 * 22050)) # 0.5 seconds of silence at 22.05kHz
            if sf: # sf might still be available if only Kokoro failed
                 sf.write(str(output_path), dummy_audio, 22050)
            else: # Fallback if soundfile is also missing
                with open(output_path, 'w') as f: f.write("dummy_wav_content_if_sf_fails") # Not a real WAV
            logger.info(f"Generated DUMMY TTS WAV file: {output_path}")
            mark_step_complete(workspace_path, "voiceover")
            return str(output_path)
        except Exception as e_dummy:
            logger.error(f"Error creating dummy WAV for TTS: {e_dummy}")
            return None

    if not KOKORO_MODEL_FILE_PATH or not KOKORO_VOICES_FILE_PATH:
        logger.error("Kokoro model or voices file path not set in environment. Cannot generate TTS.")
        return None
    if not pathlib.Path(KOKORO_MODEL_FILE_PATH).exists() or not pathlib.Path(KOKORO_VOICES_FILE_PATH).exists():
        logger.error(f"Kokoro model ({KOKORO_MODEL_FILE_PATH}) or voices ({KOKORO_VOICES_FILE_PATH}) file not found. Cannot generate TTS.")
        return None

    try:
        kokoro_tts = Kokoro(KOKORO_MODEL_FILE_PATH, KOKORO_VOICES_FILE_PATH)
        logger.info(f"Available Kokoro voices: {kokoro_tts.voices}") # Log available voices
        # Example: Use the first available English (US) voice, or a default if not found
        target_voice = next((v for v in kokoro_tts.voices if v.startswith('en-US')), kokoro_tts.voices[0] if kokoro_tts.voices else 'af_bella')
        logger.info(f"Using Kokoro voice: {target_voice}")

        samples, sample_rate = kokoro_tts.create(
            script_text,
            voice=target_voice,
            speed=1.0,
            lang="en-us" # Typically not needed if voice implies lang, but good to be explicit
        )
        sf.write(str(output_path), samples, sample_rate)
        logger.info(f"Generated TTS WAV file: {output_path}")
    except Exception as e:
        logger.error(f"Error generating TTS with Kokoro ONNX: {e}.")
        return None

    mark_step_complete(workspace_path, "voiceover")
    logger.info(f"step_04_generate_tts_kokoro completed. Output: {str(output_path)}")
    return str(output_path)
def step_05_transcribe_audio_local_whisper(audio_file_path_str: str, workspace_path: pathlib.Path) -> str | None:
    logger.info(f"Called step_05_transcribe_audio_local_whisper with audio: {audio_file_path_str}")

    # Add a small delay to ensure file system has caught up, esp. in cloud environments
    time.sleep(2)

    if not pathlib.Path(audio_file_path_str).exists():
        logger.error(f"WAV file does not exist at {audio_file_path_str} before transcription attempt!")
        return None

    output_artifact_name = "voiceover_transcription_detailed.txt"
    output_path = workspace_path / output_artifact_name
    ensure_dir_exists(output_path.parent)

    if not whisper:
        logger.warning("OpenAI Whisper library not found. Creating dummy transcript.")
        try:
            with open(output_path, 'w', encoding='utf-8') as f:
                f.write("Dummy transcript as Whisper is not available.")
            mark_step_complete(workspace_path, "voiceover_transcription_detailed")
            logger.info(f"Created DUMMY transcript file: {output_path}")
            return str(output_path)
        except Exception as e_dummy_ts:
            logger.error(f"Error creating dummy transcript: {e_dummy_ts}")
            return None

    try:
        model = whisper.load_model("base") # Consider making model choice configurable
        result = model.transcribe(audio_file_path_str, fp16=False) # fp16=False for CPU, can be True for GPU
        transcript_text = result.get("text", "")
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(transcript_text)
        logger.info(f"Created transcript file using Whisper: {output_path}")
    except Exception as e:
        logger.error(f"Error transcribing audio with Whisper for step_05: {e}.")
        return None

    mark_step_complete(workspace_path, "voiceover_transcription_detailed")
    logger.info(f"step_05_transcribe_audio_local_whisper completed. Output: {str(output_path)}")
    return str(output_path)
def step_06_correct_spelling(transcript_path_str: str, workspace_path: pathlib.Path) -> str | None:
    logger.info(f"Called step_06_correct_spelling with transcript: {transcript_path_str}")
    time.sleep(1)

    try:
        with open(transcript_path_str, 'r', encoding='utf-8') as f:
            transcript_text = f.read()
    except Exception as e:
        logger.error(f"Could not read transcript at {transcript_path_str} for step_06: {e}")
        return None

    script_json_path = workspace_path / "generated_script_gemini.json"
    original_script_text = ""
    try:
        script_data = load_json_config(script_json_path)
        if script_data and isinstance(script_data.get("script"), dict):
            original_script_text = script_data["script"].get("script", "")
    except Exception as e:
        logger.warning(f"Could not read original script at {script_json_path} for reference in step_06: {e}")

    corrected_text = transcript_text
    if SpellChecker and original_script_text: # Only attempt correction if spellchecker and original script available
        spell = SpellChecker()
        original_words = set(original_script_text.lower().split()) # Compare in lowercase
        transcript_words = transcript_text.split()
        words_to_correct = [word for word in transcript_words if word.lower() not in original_words and word.isalpha()]

        if words_to_correct:
            misspelled = spell.unknown(words_to_correct)
            temp_corrected_words = []
            word_map = {word_orig: word_orig for word_orig in transcript_words} # map original case to corrected

            for word_idx, word_val in enumerate(transcript_words):
                if word_val in misspelled:
                    corrected_version = spell.correction(word_val)
                    if corrected_version:
                        word_map[word_val] = corrected_version

            corrected_text = " ".join([word_map[w] for w in transcript_words])
            logger.info(f"Spelling correction applied. Original vs Corrected (if different):\nOriginal: {transcript_text}\nCorrected: {corrected_text}")
        else:
            logger.info("No words required spelling correction based on deviation from original script.")
    elif not SpellChecker:
        logger.warning("SpellChecker library not found. Skipping spelling correction.")
    elif not original_script_text:
        logger.warning("Original script text not available. Skipping context-aware spelling correction.")

    try:
        with open(transcript_path_str, 'w', encoding='utf-8') as f: # Overwrite with corrected/original text
            f.write(corrected_text)
        logger.info(f"Transcript (potentially corrected) saved to: {transcript_path_str}")
    except Exception as e:
        logger.error(f"Error saving corrected transcript for step_06: {e}")
        return None

    mark_step_complete(workspace_path, "corrected_transcription")
    logger.info(f"step_06_correct_spelling completed. Output: {transcript_path_str}")
    return transcript_path_str
def step_07_parse_transcript_for_image_segments(corrected_transcript_path_str: str, workspace_path: pathlib.Path) -> str | None:
    logger.info(f"Called step_07_parse_transcript_for_image_segments with transcript: {corrected_transcript_path_str}")
    time.sleep(1)

    output_artifact_name = "image_segments.json"
    output_path = workspace_path / output_artifact_name
    ensure_dir_exists(output_path.parent)

    try:
        with open(corrected_transcript_path_str, 'r', encoding='utf-8') as f:
            transcript_text = f.read().strip()
    except Exception as e:
        logger.error(f"Could not read transcript at {corrected_transcript_path_str}: {e}")
        return None

    if not transcript_text:
        logger.warning("Transcript text is empty. Cannot generate segments.")
        save_json_output(output_path, {"segments": []})
        mark_step_complete(workspace_path, "image_segments")
        return str(output_path)

    sentences = re.split(r'(?<=[.!?])\s+', transcript_text)
    sentences = [s.strip() for s in sentences if s.strip()]

    avg_words_per_sec = 2.5  # Adjust as needed, ~150 wpm
    min_segment_sec = 4.0
    max_segment_sec = 8.0

    segments = []
    current_segment_sentences = []
    current_segment_word_count = 0
    cumulative_time_offset = 0.0

    for i, sentence in enumerate(sentences):
        sentence_word_count = len(sentence.split())
        current_segment_sentences.append(sentence)
        current_segment_word_count += sentence_word_count
        estimated_duration_current_segment = current_segment_word_count / avg_words_per_sec

        # Check if current segment should be finalized
        if estimated_duration_current_segment >= min_segment_sec or i == len(sentences) - 1:
            final_segment_text = " ".join(current_segment_sentences)
            # Clamp duration, ensuring it's not zero
            final_duration = max(0.1, min(estimated_duration_current_segment, max_segment_sec))

            start_time = round(cumulative_time_offset, 2)
            end_time = round(cumulative_time_offset + final_duration, 2)

            segments.append({
                "segment_id": f"scene_001_seg_{len(segments)+1:03d}", # Scene ID can be dynamic later
                "text_segment": final_segment_text,
                "start_time": start_time,
                "end_time": end_time,
                "image_keywords": [] # To be filled by Groq in next step
            })

            cumulative_time_offset += final_duration
            current_segment_sentences = []
            current_segment_word_count = 0

    save_json_output(output_path, {"segments": segments})
    mark_step_complete(workspace_path, "image_segments")
    logger.info(f"step_07_parse_transcript_for_image_segments completed. Output: {str(output_path)}")
    return str(output_path)
GROQ_EXPECTED_SCHEMA_ITEM = {
    "type": "object",
    "properties": {
        "image_prompt": {"type": "string"},
        "keywords": {"type": "array", "items": {"type": "string"}}
    },
    "required": ["image_prompt", "keywords"]
}

def _validate_groq_item_structure(item_data):
    if not jsonschema:
        logger.warning("jsonschema not available, skipping Groq item validation.")
        return True
    try:
        jsonschema.validate(instance=item_data, schema=GROQ_EXPECTED_SCHEMA_ITEM)
        return True
    except jsonschema.ValidationError as e:
        logger.error(f"Groq item validation error: {e}")
        return False

async def _generate_image_prompts_batch_groq_helper(client, segments_batch, overall_video_title, overall_video_keywords):
    """Helper async function to process a batch of segments with Groq."""
    tasks = []
    for segment in segments_batch:
        text_segment = segment.get("text_segment", "")
        prompt = f"""
        Given the overall video title '{overall_video_title}' and keywords '{', '.join(overall_video_keywords)}',
        and the following text segment from a video script: "{text_segment}"
        Generate a concise, visually descriptive image prompt for an AI image generator (like Stable Diffusion or Midjourney).
        The prompt should capture the essence of the text segment, focusing on concrete nouns, actions, and visual style if implied.
        Also, extract 3-5 relevant keywords from this specific segment for searching stock images or refining AI prompts.
        Return ONLY a JSON object with two keys: "image_prompt" (string) and "keywords" (list of strings).
        Example JSON: {{"image_prompt": "A scientist looking at a glowing beaker in a dark lab, cinematic lighting", "keywords": ["scientist", "beaker", "lab", "glowing", "cinematic"]}}
        """
        tasks.append(client.chat.completions.create(
            messages=[{"role": "user", "content": prompt}],
            model="llama3-8b-8192", # Or other suitable model like mixtral-8x7b-32768
            temperature=0.7, # Adjust for creativity vs. literalness
            max_tokens=150, # Max tokens for the JSON output
            response_format={"type": "json_object"} # Request JSON output
        ))

    results = await asyncio.gather(*tasks, return_exceptions=True)
    processed_results = []
    for i, res in enumerate(results):
        segment_id = segments_batch[i].get("segment_id")
        original_text = segments_batch[i].get("text_segment")
        if isinstance(res, Exception):
            logger.error(f"Error generating prompt for segment {segment_id}: {res}")
            processed_results.append({"prompt_id": f"error_{segment_id}", "prompt_text": f"Error: {res}", "image_keywords": [], "original_text_segment": original_text, "segment_id": segment_id})
        else:
            try:
                groq_output_json_str = res.choices[0].message.content
                groq_output = json.loads(groq_output_json_str)
                if _validate_groq_item_structure(groq_output):
                    processed_results.append({
                        "prompt_id": segment_id,
                        "prompt_text": normalize_text(groq_output.get("image_prompt", "Default placeholder image prompt.")),
                        "image_keywords": groq_output.get("keywords", []),
                        "original_text_segment": original_text,
                        "segment_id": segment_id
                    })
                else:
                    logger.error(f"Groq output validation failed for segment {segment_id}. Raw: {groq_output_json_str}")
                    processed_results.append({"prompt_id": f"validation_err_{segment_id}", "prompt_text": "Validation error.", "image_keywords": [], "original_text_segment": original_text, "segment_id": segment_id})
            except json.JSONDecodeError as e:
                logger.error(f"JSON decode error for Groq output (segment {segment_id}): {e}. Raw: {groq_output_json_str}")
                processed_results.append({"prompt_id": f"json_err_{segment_id}", "prompt_text": "JSON decode error.", "image_keywords": [], "original_text_segment": original_text, "segment_id": segment_id})
            except Exception as e_gen:
                logger.error(f"General error processing Groq output for segment {segment_id}: {e_gen}")
                processed_results.append({"prompt_id": f"gen_err_{segment_id}", "prompt_text": "General processing error.", "image_keywords": [], "original_text_segment": original_text, "segment_id": segment_id})
    return processed_results

def step_08_generate_image_prompts_groq(image_segments_path_str: str, workspace_path: pathlib.Path, gemini_script_path_str: str | None = None) -> str | None:
    logger.info(f"Called step_08_generate_image_prompts_groq with segments: {image_segments_path_str}")
    output_artifact_name = "image_prompts_groq.json"
    output_path = workspace_path / output_artifact_name
    ensure_dir_exists(output_path.parent)

    image_segments_data = load_json_config(pathlib.Path(image_segments_path_str))
    segments = image_segments_data.get("segments", []) if image_segments_data else []

    overall_video_title = "General Video"
    overall_video_keywords = []
    if gemini_script_path_str:
        gemini_data = load_json_config(pathlib.Path(gemini_script_path_str))
        if gemini_data and isinstance(gemini_data.get("script"), dict):
            overall_video_title = gemini_data["script"].get("new_video_title", overall_video_title)
            overall_video_keywords = gemini_data["script"].get("keywords", [])

    if not Groq or not os.getenv("GROQ_API_KEY"):
        logger.warning("Groq library or API key not found. Generating dummy image prompts.")
        dummy_prompts = []
        for seg in segments:
            dummy_prompts.append({
                "prompt_id": seg.get("segment_id", uuid.uuid4().hex[:8]),
                "prompt_text": f"Dummy prompt for: {seg.get('text_segment', 'untitled segment')[:30]}...",
                "image_keywords": ["dummy", "placeholder"],
                "original_text_segment": seg.get('text_segment', ''),
                "segment_id": seg.get("segment_id")
            })
        save_json_output(output_path, {"image_prompts": dummy_prompts})
        mark_step_complete(workspace_path, "image_prompts_groq")
        return str(output_path)

    client = Groq(api_key=os.getenv("GROQ_API_KEY"))
    all_generated_prompts = []
    batch_size = 5 # Groq API might have rate limits, adjust batch size as needed

    async def run_batches():
        for i in range(0, len(segments), batch_size):
            batch = segments[i:i+batch_size]
            logger.info(f"Processing Groq prompt generation batch {i//batch_size + 1}...")
            batch_results = await _generate_image_prompts_batch_groq_helper(client, batch, overall_video_title, overall_video_keywords)
            all_generated_prompts.extend(batch_results)
            if i + batch_size < len(segments):
                 time.sleep(1) # Small delay between batches to respect potential rate limits

    # Run the async function
    # In Colab/Jupyter, asyncio.run() might cause issues if an event loop is already running.
    # Using nest_asyncio if available, or a simpler approach for basic scripts.
    try:
        import nest_asyncio
        nest_asyncio.apply()
        asyncio.run(run_batches())
    except ImportError:
        # Fallback if nest_asyncio is not installed (common in pure Python scripts)
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        try:
            loop.run_until_complete(run_batches())
        finally:
            loop.close()
            asyncio.set_event_loop(None)
    except RuntimeError as e:
        if "cannot be called from a running event loop" in str(e):
            logger.error("Asyncio runtime error. If in Jupyter/Colab, try installing nest_asyncio: !pip install nest_asyncio")
            # Fallback to sequential processing if async fails badly in environment
            # This is a simplified synchronous fallback, not ideal but better than crashing.
            logger.warning("Falling back to pseudo-synchronous Groq processing due to asyncio error.")
            for segment in segments:
                # This is NOT how the async helper is designed to be called, but for a basic fallback:
                # We'd need a synchronous version of _generate_image_prompts_batch_groq_helper
                # For now, just creating dummy prompts in this specific error case.
                 all_generated_prompts.append({"prompt_id": f"fallback_err_{segment.get('segment_id')}", "prompt_text": "Fallback due to asyncio error.", "image_keywords": [], "original_text_segment": segment.get('text_segment'), "segment_id": segment.get('segment_id')})
        else:
            raise e # Re-raise other runtime errors

    save_json_output(output_path, {"image_prompts": all_generated_prompts})
    mark_step_complete(workspace_path, "image_prompts_groq")
    logger.info(f"step_08_generate_image_prompts_groq completed. Output: {str(output_path)}")
    return str(output_path)
def _comfyui_load_workflow(workflow_file_path: str) -> dict | None:
    try:
        with open(workflow_file_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        logger.error(f"Error loading ComfyUI workflow {workflow_file_path}: {e}")
        return None

def _comfyui_queue_prompt(prompt_workflow: dict, client_id: str, server_address: str) -> dict | None:
    if not websocket:
        logger.error("websocket-client library not found, cannot connect to ComfyUI.")
        return None
    try:
        payload = {"prompt": prompt_workflow, "client_id": client_id}
        # Use http for queueing, not ws
        http_server_address = server_address.replace("ws://", "http://").replace("wss://", "https://")
        if not http_server_address.startswith(('http://', 'https://')):
            http_server_address = f"http://{http_server_address}"

        import requests # Using requests for HTTP POST
        response = requests.post(f"{http_server_address}/prompt", json=payload, timeout=20)
        response.raise_for_status() # Raise an exception for bad status codes
        return response.json()
    except requests.exceptions.RequestException as e:
        logger.error(f"Error queueing ComfyUI prompt via HTTP: {e}")
        return None
    except Exception as e:
        logger.error(f"General error in _comfyui_queue_prompt: {e}")
        return None

def _comfyui_get_image_data(filename: str, subfolder: str, image_type: str, server_address: str):
    if not websocket: return None # Should be caught earlier too
    # Use http for fetching images
    http_server_address = server_address.replace("ws://", "http://").replace("wss://", "https://")
    if not http_server_address.startswith(('http://', 'https://')):
        http_server_address = f"http://{http_server_address}"

    params = {"filename": filename, "subfolder": subfolder, "type": image_type}
    url = f"{http_server_address}/view?{urllib.parse.urlencode(params)}"
    try:
        import requests
        response = requests.get(url, timeout=20)
        response.raise_for_status()
        return response.content
    except requests.exceptions.RequestException as e:
        logger.error(f"Error fetching image {filename} from ComfyUI: {e}")
        return None

def _comfyui_get_history(prompt_id: str, server_address: str) -> dict | None:
    if not websocket: return None
    http_server_address = server_address.replace("ws://", "http://").replace("wss://", "https://")
    if not http_server_address.startswith(('http://', 'https://')):
        http_server_address = f"http://{http_server_address}"
    try:
        import requests
        response = requests.get(f"{http_server_address}/history/{prompt_id}", timeout=10)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        logger.error(f"Error fetching ComfyUI history for prompt {prompt_id}: {e}")
        return None

def _comfyui_fetch_images_from_history_ws(prompt_id: str, client_id: str, server_address: str, images_output_dir: pathlib.Path, current_prompt_text: str):
    if not websocket: return []
    ws_server_address = server_address if server_address.startswith(('ws://', 'wss://')) else f"ws://{server_address}"

    output_images_paths = []
    try:
        ws = websocket.WebSocket()
        ws.connect(f"{ws_server_address}/ws?clientId={client_id}")
        logger.info(f"ComfyUI WS connected for prompt_id: {prompt_id}")

        timeout_seconds = 60 # Max time to wait for image generation for one prompt
        start_time = time.time()
        image_received_for_prompt = False

        while True:
            if time.time() - start_time > timeout_seconds:
                logger.error(f"Timeout waiting for ComfyUI image for prompt_id {prompt_id}")
                break
            try:
                out = ws.recv()
                if isinstance(out, str):
                    message = json.loads(out)
                    if message['type'] == 'executing':
                        data = message['data']
                        if data['node'] is None and data['prompt_id'] == prompt_id:
                            logger.info(f"ComfyUI execution finished for prompt_id: {prompt_id}")
                            image_received_for_prompt = True # Assume success, will verify with history
                            break # Execution for this prompt_id is done
                    elif message['type'] == 'status':
                         logger.debug(f"ComfyUI status: {message['data']}")
                    elif message['type'] == 'progress':
                        data = message['data']
                        logger.debug(f"ComfyUI progress: {data['value']}/{data['max']}")
                    # else: logger.debug(f"ComfyUI WS message: {message}") # Other message types
                # else: logger.debug(f"ComfyUI WS binary message (ignored)") # Binary messages are not images here

            except websocket.WebSocketTimeoutException:
                logger.debug(f"ComfyUI WS timeout, retrying recv for prompt_id {prompt_id}")
                continue
            except ConnectionAbortedError as e_conn:
                 logger.error(f"ComfyUI WS ConnectionAbortedError for prompt_id {prompt_id}: {e_conn}")
                 break
            except Exception as e_ws_recv:
                logger.error(f"Error receiving from ComfyUI WS for prompt_id {prompt_id}: {e_ws_recv}")
                break
        ws.close()

        if image_received_for_prompt:
            history = _comfyui_get_history(prompt_id, server_address)
            if history and prompt_id in history:
                prompt_history = history[prompt_id]
                for node_id, node_output in prompt_history.get('outputs', {}).items():
                    if 'images' in node_output:
                        for image_data_item in node_output['images']:
                            image_bytes = _comfyui_get_image_data(image_data_item['filename'],
                                                                image_data_item.get('subfolder', ''),
                                                                image_data_item.get('type', 'output'),
                                                                server_address)
                            if image_bytes:
                                image_filename = f"{prompt_id}_{image_data_item['filename']}"
                                image_path = images_output_dir / image_filename
                                with open(image_path, 'wb') as img_f:
                                    img_f.write(image_bytes)
                                output_images_paths.append(str(image_path))
                                logger.info(f"Saved image from ComfyUI: {image_path}")
            else:
                logger.warning(f"Could not retrieve history or prompt_id {prompt_id} not in history after WS indicated completion.")
        else:
            logger.warning(f"WS loop completed for prompt {prompt_id} but no 'execution finished' signal or timeout occurred.")

    except ConnectionRefusedError:
        logger.error(f"ComfyUI WS connection refused at {ws_server_address}. Is ComfyUI running and accessible?")
    except websocket.WebSocketException as e_ws_main:
        logger.error(f"ComfyUI WebSocketException for prompt_id {prompt_id}: {e_ws_main}")
    except Exception as e_main:
        logger.error(f"General error in _comfyui_fetch_images_from_history_ws for prompt_id {prompt_id}: {e_main}")

    return output_images_paths

def _create_dummy_comfyui_images_and_manifest(prompts_list, images_output_dir, workspace_path, output_manifest_path):
    logger.warning("Pillow or websocket-client not fully available, or ComfyUI connection failed. Creating dummy images and manifest.")
    manifest_data = {"generated_images": []}
    ensure_dir_exists(images_output_dir)

    for i, prompt_info in enumerate(prompts_list):
        prompt_id = prompt_info.get("prompt_id", f"dummy_prompt_id_{i+1}")
        prompt_text = prompt_info.get("prompt_text", f"Dummy prompt for {prompt_id}")
        dummy_image_name = f"dummy_image_{prompt_id}.png"
        dummy_image_path = images_output_dir / dummy_image_name

        if Image and ImageDraw and ImageFont: # Check if Pillow components are available
            try:
                img = Image.new('RGB', (512, 512), color = (random.randint(100,200), random.randint(100,200), random.randint(100,200)))
                draw = ImageDraw.Draw(img)
                try:
                    font_path = str(ASSETS_DIR / "fonts" / "LiberationSans-Regular.ttf")
                    font_to_use = ImageFont.truetype(font_path, 20) if pathlib.Path(font_path).exists() else ImageFont.load_default()
                except IOError:
                    font_to_use = ImageFont.load_default()
                draw.text((10,10), f"Dummy for:\n{prompt_text[:50]}...", fill=(0,0,0), font=font_to_use)
                img.save(dummy_image_path, "PNG")
            except Exception as e_pil:
                logger.warning(f"Pillow is available but failed to create dummy PNG ({dummy_image_name}): {e_pil}. Creating empty file.")
                dummy_image_path.touch()
        else:
            dummy_image_path.touch() # Create empty file if Pillow is not there

        manifest_data["generated_images"].append({
            "prompt_id": prompt_id,
            "prompt_text": prompt_text,
            "image_path_local": str(dummy_image_path),
            "source_prompt_info": prompt_info
        })

    save_json_output(output_manifest_path, manifest_data)
    mark_step_complete(workspace_path, "generated_images_manifest")
    logger.info(f"Created DUMMY images manifest: {str(output_manifest_path)}")
    return str(output_manifest_path)

def step_09_generate_images_comfyui(image_prompts_path_str: str, workspace_path: pathlib.Path) -> str | None:
    logger.info(f"Called step_09_generate_images_comfyui with prompts: {image_prompts_path_str}")
    output_artifact_name = "generated_images_manifest.json"
    images_output_dir = workspace_path / "generated_images_comfyui"
    ensure_dir_exists(images_output_dir)
    output_manifest_path = workspace_path / output_artifact_name

    prompts_input = load_json_config(pathlib.Path(image_prompts_path_str))
    if not prompts_input or "image_prompts" not in prompts_input:
        logger.error(f"Invalid or empty prompts data from {image_prompts_path_str}")
        return _create_dummy_comfyui_images_and_manifest([], images_output_dir, workspace_path, output_manifest_path)
    prompts_list = prompts_input["image_prompts"]

    if not websocket or not COMFYUI_SERVER_ADDRESS or not COMFYUI_WORKFLOW_FILE or not pathlib.Path(COMFYUI_WORKFLOW_FILE).exists():
        logger.warning(f"ComfyUI prerequisites not met (websocket, server address, workflow file: {COMFYUI_WORKFLOW_FILE}). Using dummy images.")
        return _create_dummy_comfyui_images_and_manifest(prompts_list, images_output_dir, workspace_path, output_manifest_path)

    workflow_template = _comfyui_load_workflow(COMFYUI_WORKFLOW_FILE)
    if not workflow_template:
        logger.error("Failed to load ComfyUI workflow. Using dummy images.")
        return _create_dummy_comfyui_images_and_manifest(prompts_list, images_output_dir, workspace_path, output_manifest_path)

    client_id = str(uuid.uuid4())
    manifest_data = {"generated_images": []}

    for prompt_info in prompts_list:
        current_prompt_text = prompt_info.get("prompt_text", "Default positive prompt text")
        current_prompt_id = prompt_info.get("prompt_id", f"comfy_prompt_{uuid.uuid4().hex[:8]}")
        logger.info(f"Processing ComfyUI for prompt_id: {current_prompt_id}, text: '{current_prompt_text[:50]}...'" )

        # --- Modify workflow --- This part is highly dependent on your workflow structure
        # Example: find a node by title (e.g., 'Load Checkpoint') or by class_type (e.g., 'CheckpointLoaderSimple')
        # And another for the prompt text (e.g., a 'CLIPTextEncode (Prompt)' node)
        # This requires knowing your workflow's node IDs or titles.
        # For this example, let's assume node '6' is positive prompt and '7' is negative.
        # And node '4' is filename prefix for SaveImage.
        # IMPORTANT: These node IDs ('3', '6', '7', etc.) are examples and WILL need to be
        #            changed to match YOUR ComfyUI workflow.json.
        #            Inspect your workflow JSON to find the correct node IDs for text inputs and save nodes.
        current_workflow = json.loads(json.dumps(workflow_template)) # Deep copy
        try:
            # Find a KSampler node to potentially adjust seed (example)
            k_sampler_node_id = None
            for node_id, node_data in current_workflow.items():
                if node_data.get("class_type") == "KSampler":
                    k_sampler_node_id = node_id
                    break
            if k_sampler_node_id:
                 current_workflow[k_sampler_node_id]["inputs"]["seed"] = random.randint(0, 2**32 - 1)

            # Find positive prompt node (example: by title, then by common class_type)
            positive_prompt_node_id = None
            for node_id, node_data in current_workflow.items():
                 # Try matching by a common title if users name their nodes
                if node_data.get("_meta", {}).get("title") == "Positive Prompt Text":
                    positive_prompt_node_id = node_id
                    break
            if not positive_prompt_node_id: # Fallback to class_type if not found by title
                for node_id, node_data in current_workflow.items():
                    if node_data.get("class_type") == "CLIPTextEncode": # Common for prompts
                        # This is a heuristic; might pick the wrong one if multiple exist.
                        # A more robust method is needed if titles aren't used (e.g. specific node ID from user config)
                        positive_prompt_node_id = node_id
                        break # Take the first one found for this example

            if positive_prompt_node_id:
                current_workflow[positive_prompt_node_id]["inputs"]["text"] = current_prompt_text
            else:
                logger.warning(f"Could not find a suitable positive prompt node in ComfyUI workflow for prompt_id {current_prompt_id}. Prompt not set.")

            # Find SaveImage node to set filename_prefix
            save_image_node_id = None
            for node_id, node_data in current_workflow.items():
                if node_data.get("class_type") == "SaveImage":
                    save_image_node_id = node_id
                    break
            if save_image_node_id:
                current_workflow[save_image_node_id]["inputs"]["filename_prefix"] = f"img_{current_prompt_id}"
            else:
                 logger.warning(f"Could not find SaveImage node in ComfyUI workflow for prompt_id {current_prompt_id}. Filename prefix not set.")

        except KeyError as e:
            logger.error(f"KeyError modifying ComfyUI workflow for prompt {current_prompt_id} (node ID {e} likely missing/incorrect). Skipping this prompt.")
            # Add dummy entry for this failed prompt
            manifest_data["generated_images"].append({
                "prompt_id": current_prompt_id,
                "prompt_text": current_prompt_text,
                "image_path_local": "error_workflow_modification.png",
                "source_prompt_info": prompt_info,
                "error": f"Workflow modification failed: KeyError {e}"
            })
            continue # Move to the next prompt in prompts_list
        except Exception as e_wf_mod:
            logger.error(f"Unexpected error modifying ComfyUI workflow for prompt {current_prompt_id}: {e_wf_mod}. Skipping this prompt.")
            manifest_data["generated_images"].append({
                "prompt_id": current_prompt_id,
                "prompt_text": current_prompt_text,
                "image_path_local": "error_workflow_modification_unexpected.png",
                "source_prompt_info": prompt_info,
                "error": f"Workflow modification failed: {e_wf_mod}"
            })
            continue

        queued_prompt_info = _comfyui_queue_prompt(current_workflow, client_id, COMFYUI_SERVER_ADDRESS)
        if queued_prompt_info and 'prompt_id' in queued_prompt_info:
            comfy_prompt_run_id = queued_prompt_info['prompt_id']
            logger.info(f"ComfyUI prompt {current_prompt_id} queued successfully. ComfyUI run ID: {comfy_prompt_run_id}")
            # Fetch images using the ComfyUI run ID (which is also a prompt_id in their system)
            image_paths = _comfyui_fetch_images_from_history_ws(comfy_prompt_run_id, client_id, COMFYUI_SERVER_ADDRESS, images_output_dir, current_prompt_text)
            if image_paths:
                for img_path in image_paths: # Typically one image per prompt unless workflow saves multiple
                    manifest_data["generated_images"].append({
                        "prompt_id": current_prompt_id, # Link back to original segment/prompt ID
                        "comfyui_run_id": comfy_prompt_run_id,
                        "prompt_text": current_prompt_text,
                        "image_path_local": img_path,
                        "source_prompt_info": prompt_info
                    })
            else:
                logger.warning(f"No images saved from ComfyUI for prompt_id {current_prompt_id} (Comfy run ID {comfy_prompt_run_id}). Check ComfyUI logs/status.")
                # Add placeholder if image fetching failed but queueing was ok
                manifest_data["generated_images"].append({
                    "prompt_id": current_prompt_id,
                    "comfyui_run_id": comfy_prompt_run_id,
                    "prompt_text": current_prompt_text,
                    "image_path_local": "error_comfy_image_fetch.png",
                    "source_prompt_info": prompt_info,
                    "error": "Failed to fetch image from ComfyUI after queueing."
                })
        else:
            logger.error(f"Failed to queue prompt in ComfyUI for prompt_id: {current_prompt_id}. Check ComfyUI server connection and workflow.")
            manifest_data["generated_images"].append({
                "prompt_id": current_prompt_id,
                "prompt_text": current_prompt_text,
                "image_path_local": "error_comfy_queueing.png",
                "source_prompt_info": prompt_info,
                "error": "Failed to queue prompt in ComfyUI."
            })
        time.sleep(1) # Small delay between prompts if rapidly queueing

    if not manifest_data["generated_images"]:
        logger.warning("No images were successfully generated or processed by ComfyUI. Creating dummy manifest.")
        return _create_dummy_comfyui_images_and_manifest(prompts_list, images_output_dir, workspace_path, output_manifest_path)

    save_json_output(output_manifest_path, manifest_data)
    mark_step_complete(workspace_path, "generated_images_manifest")
    logger.info(f"step_09_generate_images_comfyui completed. Manifest: {str(output_manifest_path)}")
    return str(output_manifest_path)
def _ease_in_quad(t: float) -> float:
    return t * t

def _ease_out_quad(t: float) -> float:
    return t * (2 - t)

def _ease_in_out_quad(t: float) -> float:
    if t < 0.5:
        return 2 * t * t
    return 1 - pow(-2 * t + 2, 2) / 2

ANIM_SCREEN_W, ANIM_SCREEN_H = 1024, 576
ANIM_FPS = 24

def _prepare_image_for_animation(img_clip_orig: 'ImageClip', target_canvas_size: tuple[int, int], cover_scale_factor: float = 1.0) -> 'ImageClip | None':
    if not ImageClip_cls or not img_clip_orig:
        logger.error("ImageClip_cls not available or img_clip_orig is None in _prepare_image_for_animation")
        return None

    img_w, img_h = img_clip_orig.size
    canvas_w, canvas_h = target_canvas_size

    cover_w = canvas_w * cover_scale_factor
    cover_h = canvas_h * cover_scale_factor

    aspect_img = img_w / img_h
    aspect_cover = cover_w / cover_h

    if aspect_img > aspect_cover:
        new_h = cover_h
        new_w = int(new_h * aspect_img)
    else:
        new_w = cover_w
        new_h = int(new_w / aspect_img)

    # Ensure new_w and new_h are at least 1 to avoid issues with resize
    new_w = max(1, int(new_w))
    new_h = max(1, int(new_h))

    return img_clip_orig.resize((new_w, new_h))

def _animate_static(img_clip_prepared: 'ImageClip', duration: float, target_size: tuple[int,int], **kwargs) -> 'CompositeVideoClip | None':
    if not moviepy_ok or not ColorClip_cls or not CompositeVideoClip_cls or not img_clip_prepared:
        logger.error("MoviePy components missing or invalid image clip for static animation.")
        return None
    bg = ColorClip_cls(size=target_size, color=(0,0,0), duration=duration, ismask=False).set_fps(ANIM_FPS)
    img_to_display = img_clip_prepared.copy()
    if img_to_display.w > target_size[0] or img_to_display.h > target_size[1]:
        resize_ratio_w = target_size[0] / img_to_display.w
        resize_ratio_h = target_size[1] / img_to_display.h
        if resize_ratio_w < resize_ratio_h:
            img_to_display = img_to_display.resize(width=target_size[0])
        else:
            img_to_display = img_to_display.resize(height=target_size[1])
    return CompositeVideoClip_cls([bg, img_to_display.set_position('center')], size=target_size).set_duration(duration)

def _animate_zoom_in(img_clip_prepared: 'ImageClip', duration: float, target_size: tuple[int,int], zoom_factor: float = 1.2, ease_func=_ease_in_out_quad, **kwargs) -> 'CompositeVideoClip | None':
    if not moviepy_ok or not ColorClip_cls or not CompositeVideoClip_cls or not img_clip_prepared:
        logger.error("MoviePy components missing or invalid image clip for zoom-in animation.")
        return None
    bg = ColorClip_cls(size=target_size, color=(0,0,0), duration=duration, ismask=False, fps=ANIM_FPS)
    animated_img = img_clip_prepared.resize(lambda t: 1 + (zoom_factor - 1) * ease_func(t / duration))
    return CompositeVideoClip_cls([bg, animated_img.set_position('center')], size=target_size).set_duration(duration)

def _animate_zoom_out(img_clip_prepared: 'ImageClip', duration: float, target_size: tuple[int,int], zoom_factor: float = 1.2, ease_func=_ease_in_out_quad, **kwargs) -> 'CompositeVideoClip | None':
    if not moviepy_ok or not ColorClip_cls or not CompositeVideoClip_cls or not img_clip_prepared:
        logger.error("MoviePy components missing or invalid image clip for zoom-out animation.")
        return None
    bg = ColorClip_cls(size=target_size, color=(0,0,0), duration=duration, ismask=False, fps=ANIM_FPS)
    animated_img = img_clip_prepared.resize(lambda t: 1 - (1 - 1/zoom_factor) * ease_func(t / duration))
    return CompositeVideoClip_cls([bg, animated_img.set_position('center')], size=target_size).set_duration(duration)

def _animate_pan(img_clip_prepared: 'ImageClip', duration: float, target_size: tuple[int,int], direction: str = "left", ease_func=_ease_in_out_quad, **kwargs) -> 'CompositeVideoClip | None':
    if not moviepy_ok or not ColorClip_cls or not CompositeVideoClip_cls or not img_clip_prepared:
        logger.error("MoviePy components missing or invalid image clip for pan animation.")
        return None
    bg = ColorClip_cls(size=target_size, color=(0,0,0), duration=duration, ismask=False, fps=ANIM_FPS)
    img_w, img_h = img_clip_prepared.size
    screen_w, screen_h = target_size
    start_x, start_y, end_x, end_y = 0.0, 0.0, 0.0, 0.0

    if direction == "left": start_x, end_x = 0, -(img_w - screen_w); start_y = end_y = 'center'
    elif direction == "right": start_x, end_x = -(img_w - screen_w), 0; start_y = end_y = 'center'
    elif direction == "up": start_y, end_y = 0, -(img_h - screen_h); start_x = end_x = 'center'
    elif direction == "down": start_y, end_y = -(img_h - screen_h), 0; start_x = end_x = 'center'
    else: return _animate_static(img_clip_prepared, duration, target_size) # Fallback

    if (direction in ["left", "right"] and img_w <= screen_w + 1) or (direction in ["up", "down"] and img_h <= screen_h + 1):
        logger.warning(f"Cannot pan {direction} as image size ({img_w}x{img_h}) not sufficiently larger than screen ({screen_w}x{screen_h}). Using static.")
        return _animate_static(img_clip_prepared, duration, target_size)

    def pos_func(t):
        e_t = ease_func(t / duration)
        curr_x = start_x if isinstance(start_x, str) else start_x + (end_x - start_x) * e_t
        curr_y = start_y if isinstance(start_y, str) else start_y + (end_y - start_y) * e_t
        return (int(curr_x) if isinstance(curr_x, (float, int)) else curr_x, int(curr_y) if isinstance(curr_y, (float, int)) else curr_y)

    animated_img = img_clip_prepared.set_position(pos_func)
    return CompositeVideoClip_cls([bg, animated_img], size=target_size).set_duration(duration)

def _animate_diag_pan_zoom_in(img_clip_prepared: 'ImageClip', duration: float, target_size: tuple[int,int], zoom_factor: float = 1.2, ease_func=_ease_in_out_quad, direction: tuple[str,str]=('left', 'up'), **kwargs) -> 'CompositeVideoClip | None':
    if not moviepy_ok or not ColorClip_cls or not CompositeVideoClip_cls or not img_clip_prepared:
        logger.error("MoviePy components missing or invalid image clip for diagonal pan-zoom animation.")
        return None
    bg = ColorClip_cls(size=target_size, color=(0,0,0), duration=duration, ismask=False, fps=ANIM_FPS)
    current_scale_lambda = lambda t: 1 + (zoom_factor - 1) * ease_func(t / duration)
    pan_avail_x_orig = max(0, img_clip_prepared.w - target_size[0])
    pan_avail_y_orig = max(0, img_clip_prepared.h - target_size[1])
    start_x_tl, end_x_tl = 0.0, 0.0
    start_y_tl, end_y_tl = 0.0, 0.0
    if direction[0] == 'left': start_x_tl, end_x_tl = 0, -pan_avail_x_orig
    elif direction[0] == 'right': start_x_tl, end_x_tl = -pan_avail_x_orig, 0
    if direction[1] == 'up': start_y_tl, end_y_tl = 0, -pan_avail_y_orig
    elif direction[1] == 'down': start_y_tl, end_y_tl = -pan_avail_y_orig, 0

    # If not enough room to pan in chosen direction with current overscale, fall back to simpler zoom or static
    can_pan_x = pan_avail_x_orig > 1 and direction[0] in ['left', 'right']
    can_pan_y = pan_avail_y_orig > 1 and direction[1] in ['up', 'down']
    if (direction[0] != 'center' and not can_pan_x) or (direction[1] != 'center' and not can_pan_y):
        logger.warning(f"Cannot effectively pan diagonally for {direction} with image size {img_clip_prepared.size} vs screen {target_size}. Falling back to zoom_in.")
        return _animate_zoom_in(img_clip_prepared, duration, target_size, zoom_factor=zoom_factor, ease_func=ease_func)

    def pos_func_diag(t):
        e_t = ease_func(t / duration)
        curr_x = start_x_tl + (end_x_tl - start_x_tl) * e_t
        curr_y = start_y_tl + (end_y_tl - start_y_tl) * e_t
        return (int(curr_x), int(curr_y))
    animated_img = img_clip_prepared.resize(current_scale_lambda).set_position(pos_func_diag)
    return CompositeVideoClip_cls([bg, animated_img], size=target_size).set_duration(duration)

def _animate_zoom_in_fade_in(img_clip_prepared: 'ImageClip', duration: float, target_size: tuple[int,int], zoom_factor: float = 1.2, fade_duration_ratio: float = 0.25, ease_func=_ease_in_out_quad, **kwargs) -> 'CompositeVideoClip | None':
    if not moviepy_ok or not moviepy_vfx_all or not img_clip_prepared:
        logger.warning("Fade-in effect not available or invalid image. Performing zoom-in only.")
        return _animate_zoom_in(img_clip_prepared, duration, target_size, zoom_factor=zoom_factor, ease_func=ease_func)
    zoomed_composite_clip = _animate_zoom_in(img_clip_prepared, duration, target_size, zoom_factor=zoom_factor, ease_func=ease_func)
    if not zoomed_composite_clip: return None
    fade_duration_actual = duration * fade_duration_ratio
    return zoomed_composite_clip.fx(moviepy_vfx_all.fadein, duration=fade_duration_actual)

def _animate_rotate_zoom(img_clip_prepared: 'ImageClip', duration: float, target_size: tuple[int,int], angle_deg: float = 10, zoom_factor: float = 1.1, ease_func=_ease_in_out_quad, **kwargs) -> 'CompositeVideoClip | None':
    if not moviepy_ok or not moviepy_vfx_all or not ColorClip_cls or not CompositeVideoClip_cls or not img_clip_prepared:
        logger.warning("Rotate effect not available or invalid image. Performing zoom-in only.")
        return _animate_zoom_in(img_clip_prepared, duration, target_size, zoom_factor=zoom_factor, ease_func=ease_func)
    bg = ColorClip_cls(size=target_size, color=(0,0,0), duration=duration, ismask=False, fps=ANIM_FPS)
    angle_lambda = lambda t: ease_func(t / duration) * angle_deg
    scale_lambda = lambda t: 1 + (zoom_factor - 1) * ease_func(t / duration)
    animated_img = img_clip_prepared.fx(moviepy_vfx_all.rotate, angle_lambda, expand=False, resample='bilinear')
    animated_img = animated_img.resize(scale_lambda)
    return CompositeVideoClip_cls([bg, animated_img.set_position('center')], size=target_size).set_duration(duration)

ANIMATION_RECIPES = {
    "static": _animate_static,
    "zoom_in": _animate_zoom_in,
    "zoom_out": _animate_zoom_out,
    "pan_left": lambda ic, d, ts, **k: _animate_pan(ic, d, ts, direction="left", **k),
    "pan_right": lambda ic, d, ts, **k: _animate_pan(ic, d, ts, direction="right", **k),
    "pan_up": lambda ic, d, ts, **k: _animate_pan(ic, d, ts, direction="up", **k),
    "pan_down": lambda ic, d, ts, **k: _animate_pan(ic, d, ts, direction="down", **k),
    "diag_pan_zoom_in_lu": lambda ic, d, ts, **k: _animate_diag_pan_zoom_in(ic, d, ts, direction=('left', 'up'), **k),
    "diag_pan_zoom_in_ld": lambda ic, d, ts, **k: _animate_diag_pan_zoom_in(ic, d, ts, direction=('left', 'down'), **k),
    "diag_pan_zoom_in_ru": lambda ic, d, ts, **k: _animate_diag_pan_zoom_in(ic, d, ts, direction=('right', 'up'), **k),
    "diag_pan_zoom_in_rd": lambda ic, d, ts, **k: _animate_diag_pan_zoom_in(ic, d, ts, direction=('right', 'down'), **k),
    "zoom_fade_in": _animate_zoom_in_fade_in,
    "rotate_zoom": _animate_rotate_zoom,
}
EASING_FUNCTIONS_LIST = [_ease_in_quad, _ease_out_quad, _ease_in_out_quad]
ANIMATION_COVER_SCALES = {
    "static": 1.0, "zoom_in": 1.0, "zoom_out": 1.2, "pan_left": 1.25, "pan_right": 1.25,
    "pan_up": 1.25, "pan_down": 1.25, "diag_pan_zoom_in_lu": 1.25, "diag_pan_zoom_in_ld": 1.25,
    "diag_pan_zoom_in_ru": 1.25, "diag_pan_zoom_in_rd": 1.25, "zoom_fade_in": 1.0, "rotate_zoom": 1.5,
}
def step_10_animate_images(generated_images_manifest_path_str: str, image_segments_path_str: str, workspace_path: pathlib.Path) -> str | None:
    logger.info(f"Starting step_10_animate_images from manifest: {generated_images_manifest_path_str}")
    if not moviepy_ok or not ImageClip_cls:
        logger.error("MoviePy or ImageClip not available. Cannot animate images.")
        # Create an empty manifest to allow pipeline to proceed if fallbacks are desired later
        output_manifest_path_dummy = workspace_path / "animated_clips_manifest.json"
        save_json_output(output_manifest_path_dummy, {"animated_clips": [], "status": "error_moviepy_missing"})
        mark_step_complete(workspace_path, "animated_clips_manifest") # Mark to avoid re-run
        return str(output_manifest_path_dummy)

    artifact_name = "animated_clips_manifest.json"
    clips_output_dir_name = "animated_image_clips"
    output_manifest_path = workspace_path / artifact_name
    clips_dir = workspace_path / clips_output_dir_name
    ensure_dir_exists(clips_dir)

    images_manifest = load_json_config(pathlib.Path(generated_images_manifest_path_str))
    segments_manifest = load_json_config(pathlib.Path(image_segments_path_str))

    if not images_manifest or "generated_images" not in images_manifest or not images_manifest["generated_images"]:
        logger.warning("No generated images found in manifest for animation. Creating empty animation manifest.")
        save_json_output(output_manifest_path, {"animated_clips": [], "status": "no_images_to_animate"})
        mark_step_complete(workspace_path, "animated_clips_manifest")
        return str(output_manifest_path)

    if not segments_manifest or "segments" not in segments_manifest:
        logger.error("Segments manifest is missing or invalid. Cannot determine animation durations.")
        # Create an empty manifest as we can't proceed
        save_json_output(output_manifest_path, {"animated_clips": [], "status": "error_segments_missing"})
        mark_step_complete(workspace_path, "animated_clips_manifest")
        return str(output_manifest_path)

    # Create a lookup for segments by prompt_id (which should match segment_id from Groq step)
    segment_duration_map = {seg.get("segment_id"): (seg.get("end_time", 0) - seg.get("start_time", 0))
                            for seg in segments_manifest["segments"]}

    animated_clips_output_data = []
    available_animations = list(ANIMATION_RECIPES.keys())

    for idx, img_info in enumerate(images_manifest["generated_images"]):
        image_path_str = img_info.get("image_path_local")
        prompt_id = img_info.get("prompt_id") # This should map to a segment_id

        if not image_path_str or not pathlib.Path(image_path_str).exists():
            logger.warning(f"Image path {image_path_str} for prompt {prompt_id} not found or invalid. Skipping animation.")
            animated_clips_output_data.append({
                "prompt_id": prompt_id,
                "animation_type": "error_image_missing",
                "clip_path": None,
                "duration": 0,
                "original_image_path": image_path_str
            })
            continue

        # Determine duration from segments manifest using prompt_id
        # The prompt_id from ComfyUI manifest (linked to Groq output) should match segment_id from segment parse step
        duration = segment_duration_map.get(prompt_id, (4.0 + 8.0)/2) # Default if not found
        if duration <= 0: duration = 4.0 # Ensure positive duration

        try:
            img_clip_orig = ImageClip_cls(image_path_str).set_duration(duration)
        except Exception as e:
            logger.error(f"Failed to load image {image_path_str} with MoviePy: {e}. Skipping animation.")
            animated_clips_output_data.append({
                "prompt_id": prompt_id,
                "animation_type": "error_image_load_moviepy",
                "clip_path": None,
                "duration": duration,
                "original_image_path": image_path_str
            })
            continue

        # Randomly select animation type and easing function
        anim_type_name = random.choice(available_animations)
        ease_func_selected = random.choice(EASING_FUNCTIONS_LIST)
        cover_scale = ANIMATION_COVER_SCALES.get(anim_type_name, 1.0)

        img_clip_prepared = _prepare_image_for_animation(img_clip_orig, (ANIM_SCREEN_W, ANIM_SCREEN_H), cover_scale_factor=cover_scale)
        if not img_clip_prepared:
            logger.error(f"Failed to prepare image {image_path_str} for animation. Skipping.")
            # Log error and skip to next image
            animated_clips_output_data.append({
                "prompt_id": prompt_id,
                "animation_type": "error_prepare_failed",
                "clip_path": None,
                "duration": duration,
                "original_image_path": image_path_str
            })
            img_clip_orig.close() # Close original clip
            continue

        logger.info(f"Animating {prompt_id} ({idx+1}/{len(images_manifest['generated_images'])}) with '{anim_type_name}', duration: {duration:.2f}s, cover_scale: {cover_scale}")

        animation_func = ANIMATION_RECIPES[anim_type_name]
        # Pass specific kwargs if needed by certain animations, e.g., zoom_factor for zoom animations
        anim_kwargs = {'ease_func': ease_func_selected, 'zoom_factor': random.uniform(1.1, 1.3)}
        if 'rotate' in anim_type_name: anim_kwargs['angle_deg'] = random.uniform(-10, 10)

        final_clip = None
        try:
            final_clip = animation_func(img_clip_prepared, duration, (ANIM_SCREEN_W, ANIM_SCREEN_H), **anim_kwargs)
        except Exception as e_anim:
            logger.error(f"Error during animation '{anim_type_name}' for {prompt_id}: {e_anim}. Trying static fallback.")
            # Fallback to static animation if the chosen one fails
            try:
                # Reprepare for static if original prep was too large (e.g. for rotate)
                static_prepared_clip = _prepare_image_for_animation(img_clip_orig, (ANIM_SCREEN_W, ANIM_SCREEN_H), cover_scale_factor=1.0)
                if static_prepared_clip:
                    final_clip = _animate_static(static_prepared_clip, duration, (ANIM_SCREEN_W, ANIM_SCREEN_H))
                    anim_type_name = "static_fallback"
                else:
                    logger.error(f"Fallback static preparation also failed for {prompt_id}.")
            except Exception as e_static_fallback:
                 logger.error(f"Error during static fallback animation for {prompt_id}: {e_static_fallback}.")

        output_video_path = None
        if final_clip:
            try:
                output_video_filename = f"anim_{prompt_id}.mp4"
                output_video_path_obj = clips_dir / output_video_filename
                # Use specified FPS, ensure audio is False if no audio track is intended for these image animations
                final_clip.write_videofile(str(output_video_path_obj), fps=ANIM_FPS, codec="libx264", audio=False, logger=None) # threads=4, preset='medium'
                output_video_path = str(output_video_path_obj)
                logger.info(f"Saved animated clip: {output_video_path}")
            except Exception as e_write:
                logger.error(f"Error writing video file for {prompt_id}: {e_write}")
                output_video_path = None # Ensure path is None if write fails
            finally:
                final_clip.close() # Close the final_clip
        else:
            logger.warning(f"No final_clip generated for {prompt_id} (animation failed). Not writing video file.")

        animated_clips_output_data.append({
            "prompt_id": prompt_id,
            "animation_type": anim_type_name,
            "clip_path": output_video_path, # Store None if write failed or no clip
            "duration": duration,
            "original_image_path": image_path_str
        })

        # Clean up MoviePy clips
        img_clip_orig.close()
        if img_clip_prepared: img_clip_prepared.close()
        # Small delay to manage resources, especially if many clips
        time.sleep(0.1)

    save_json_output(output_manifest_path, {"animated_clips": animated_clips_output_data})
    mark_step_complete(workspace_path, "animated_clips_manifest")
    logger.info(f"step_10_animate_images completed. Output manifest: {output_manifest_path}")
    return str(output_manifest_path)


# --- Main Pipeline Execution Function ---
# --- Main Pipeline Execution Function ---
def main():
    # The INPUT_DIR, WORKSPACE_DIR etc. are already defined at the top.

    # === CHOOSE YOUR INPUT TYPE AND PATH ===
    # Option 1: Text input (e.g., a JSON file with a script or article)
    # Ensure 'text_sources' sub-directory exists in your INPUT_DIR on Drive.
    # This path correctly points to your file in Google Drive: /content/drive/MyDrive/VideoPipelineProject/input/text_sources/text1.json
    input_source_filename = "text1.json" # Specify the input file name
    input_source_path = str(INPUT_DIR / "text_sources" / input_source_filename)

    # --- Validate Input File Exists ---
    if not pathlib.Path(input_source_path).exists():
        logger.error(f"Input file {input_source_path} does not exist. Please check the path and ensure the file is in your Google Drive.")
        # In Colab, raising an exception is better than sys.exit()
        raise FileNotFoundError(f"Input file {input_source_path} not found.")

    # Generate a unique ID for this run based on the input file name
    # This creates a dynamic workspace for each run based on the input filename
    source_id = get_source_id(input_source_filename)
    current_workspace_path = get_workspace_path(source_id)
    ensure_dir_exists(current_workspace_path)
    logger.info(f"Using workspace: {current_workspace_path}")

    # --- Step 1: Process text input ---
    # Pass the full, correctly constructed path to step_01
    processed_text_path = step_01_process_text_input(input_source_path, current_workspace_path)

    # --- Load and Print Processed Text Output from Step 1 ---
    if processed_text_path:
        step1_output_data = load_json_config(pathlib.Path(processed_text_path))
        if step1_output_data and step1_output_data.get("status") == "success":
            print("\n--- Step 1 Processed Text Output ---")
            print(step1_output_data.get("processed_content", "Content not found in Step 1 output."))
            print("------------------------------------\n")
        elif step1_output_data:
             logger.error(f"Step 1 reported status: {step1_output_data.get('status')}. Processed content may be empty or indicate an error.")
        else:
            logger.error("Failed to load output JSON from Step 1.")

    # Check if Step 1 was successful before proceeding
    if not processed_text_path:
        logger.error("Step 1 (Process Text Input) failed. Cannot proceed with the rest of the pipeline.")
        raise Exception("Pipeline stopped due to Step 1 failure.")
    logger.info(f"Step 1 output file: {processed_text_path}")



        # Option 2: Video link input (e.g., a text file containing a YouTube URL)
    # Ensure 'link_sources' sub-directory exists in your INPUT_DIR on Drive.
    # Example: /content/drive/MyDrive/VideoPipelineProject/input/link_sources/youtube_video.txt
    # input_source_filename = "my_video_url.txt" # MODIFY FILENAME AS NEEDED
    # input_source_path = str(INPUT_DIR / "link_sources" / input_source_filename)
    # input_type = "video_url"
    # =======================================

    # --- (Optional) Step 2: Process Video Link (if applicable) ---
    # Since the primary input is direct text, Step 2 related to video links from files
    # is irrelevant and can be skipped entirely or modified if you need supplementary video info.
    logger.info("Skipping Step 2 as primary input is direct text.")
    # If you had a separate video URL to process for supplementary info, you would call step_02 here
    # downloaded_video_info_path = step_02_process_video_link_input("YOUR_VIDEO_URL_HERE", current_workspace_path)


    # --- Step 3: Generate Script (Gemini) ---
    if not os.getenv("GEMINI_API_KEY"):
        logger.error("GEMINI_API_KEY not set. Cannot proceed with Step 3.")
        raise ValueError("Missing GEMINI_API_KEY for Step 3.")
    gemini_script_path = step_03_generate_gemini_script(processed_text_path, current_workspace_path)
    if not gemini_script_path:
        logger.error("Step 3 (Generate Gemini Script) failed.")
        raise Exception("Step 3 failed.")
    logger.info(f"Step 3 output: {gemini_script_path}")

    # --- Step 4: Generate TTS (Kokoro) ---
    if not KOKORO_AVAILABLE or not KOKORO_MODEL_FILE_PATH or not KOKORO_VOICES_FILE_PATH or \
       not pathlib.Path(KOKORO_MODEL_FILE_PATH).exists() or not pathlib.Path(KOKORO_VOICES_FILE_PATH).exists():
        logger.warning(f"Kokoro TTS prerequisites not met (KOKORO_AVAILABLE={KOKORO_AVAILABLE}, model_path='{KOKORO_MODEL_FILE_PATH}', voices_path='{KOKORO_VOICES_FILE_PATH}'). Attempting to use dummy TTS output if enabled, or step will fail.")
        # The step_04 function itself handles creating a dummy if KOKORO_AVAILABLE is False.
        # If paths are missing but KOKORO_AVAILABLE was True, it will fail inside the step.

    tts_audio_path = step_04_generate_tts_kokoro(gemini_script_path, current_workspace_path)
    if not tts_audio_path:
        logger.error("Step 4 (Generate TTS) failed.")
        raise Exception("Step 4 failed.")
    logger.info(f"Step 4 output: {tts_audio_path}")

    # --- Step 5: Transcribe Audio (Whisper) ---
    transcribed_text_path = step_05_transcribe_audio_local_whisper(tts_audio_path, current_workspace_path)
    if not transcribed_text_path:
        logger.error("Step 5 (Transcribe Audio) failed.")
        raise Exception("Step 5 failed.")
    logger.info(f"Step 5 output: {transcribed_text_path}")

    # --- Step 6: Correct Spelling ---
    corrected_transcript_path = step_06_correct_spelling(transcribed_text_path, current_workspace_path)
    if not corrected_transcript_path:
        logger.error("Step 6 (Correct Spelling) failed.")
        raise Exception("Step 6 failed.")
    logger.info(f"Step 6 output: {corrected_transcript_path}")

    # --- Step 7: Parse Transcript for Image Segments ---
    image_segments_path = step_07_parse_transcript_for_image_segments(corrected_transcript_path, current_workspace_path)
    if not image_segments_path:
        logger.error("Step 7 (Parse Transcript for Image Segments) failed.")
        raise Exception("Step 7 failed.")
    logger.info(f"Step 7 output: {image_segments_path}")

    # --- Step 8: Generate Image Prompts (Groq) ---
    if not os.getenv("GROQ_API_KEY"):
        logger.error("GROQ_API_KEY not set. Cannot proceed with Step 8.")
        raise ValueError("Missing GROQ_API_KEY for Step 8.")
    # Pass gemini_script_path for overall title/keywords context
    image_prompts_path = step_08_generate_image_prompts_groq(image_segments_path, current_workspace_path, gemini_script_path)
    if not image_prompts_path:
        logger.error("Step 8 (Generate Image Prompts) failed.")
        raise Exception("Step 8 failed.")
    logger.info(f"Step 8 output: {image_prompts_path}")

    # --- Step 9: Generate Images (ComfyUI) ---
    if not COMFYUI_SERVER_ADDRESS or not COMFYUI_WORKFLOW_FILE or not pathlib.Path(COMFYUI_WORKFLOW_FILE).exists():
        logger.warning(f"ComfyUI server, workflow file, or its existence not properly configured (Server: '{COMFYUI_SERVER_ADDRESS}', Workflow: '{COMFYUI_WORKFLOW_FILE}'). Step 9 will likely use dummy images.")
        # The step_09 function handles creating dummy images if needed.

    images_manifest_path = step_09_generate_images_comfyui(image_prompts_path, current_workspace_path)
    if not images_manifest_path:
        logger.error("Step 9 (Generate Images) failed.")
        raise Exception("Step 9 failed.")
    logger.info(f"Step 9 output: {images_manifest_path}")

    # --- Step 10: Animate Images ---
    animated_clips_manifest_path = step_10_animate_images(images_manifest_path, image_segments_path, current_workspace_path)
    if not animated_clips_manifest_path:
        logger.error("Step 10 (Animate Images) failed.")
        raise Exception("Step 10 failed.")
    logger.info(f"Step 10 output: {animated_clips_manifest_path}")


    # --- Step 11, 12, 13, 14, 15 would follow here ---
    # For example, to call a placeholder Step 11 (Assemble Video):
    # final_video_manifest_path = step_11_assemble_video(animated_clips_manifest_path, tts_audio_path, current_workspace_path)
    # if not final_video_manifest_path:
    #     logger.error("Step 11 (Assemble Video) failed.")
    #     raise Exception("Step 11 failed.")
    # logger.info(f"Step 11 output: {final_video_manifest_path}")

    cprint("\n🎉🎉🎉 Video Pipeline Processing Completed (up to implemented steps)! 🎉🎉🎉", "green", attrs=["bold"])
    logger.info(f"Final animated clips manifest: {animated_clips_manifest_path}")
    logger.info(f"All intermediate files are in workspace: {current_workspace_path}")
    # The user would then look into current_workspace_path / "final_video_assembly" for outputs of later steps.

if __name__ == "__main__":
    main()
    # The user would then look into current_workspace_path / "final_video_assembly" for outputs of later steps.

Input directory: /content/drive/MyDrive/VideoPipelineProject/input
Workspace directory: /content/drive/MyDrive/VideoPipelineProject/workspace
Assets directory: /content/drive/MyDrive/VideoPipelineProject/assets
ComfyUI Server Address: 127.0.0.1:8188
ComfyUI Workflow File: /content/drive/MyDrive/VideoPipelineProject/assets/default_comfyui_workflow.json
Kokoro Model Path: /content/drive/MyDrive/VideoPipelineProject/assets/kokoro_model.onnx
Kokoro Voices Path: /content/drive/MyDrive/VideoPipelineProject/assets/kokoro_voices.json
Endscreen Video File: /content/drive/MyDrive/VideoPipelineProject/assets/default_endscreen.mp4
INFO: Kokoro-ONNX and soundfile imported successfully for TTS.
INFO: MoviePy library and components loaded successfully.

--- Step 1 Processed Text Output ---
The Moon is Earth's only natural satellite and is the fifth largest moon in the Solar System.
------------------------------------



ERROR:__main__:Kokoro model (/content/drive/MyDrive/VideoPipelineProject/assets/kokoro_model.onnx) or voices (/content/drive/MyDrive/VideoPipelineProject/assets/kokoro_voices.json) file not found. Cannot generate TTS.
ERROR:__main__:Step 4 (Generate TTS) failed.


Exception: Step 4 failed.

### Instructions to Run:
1.  **Complete Setup**: Ensure all setup cells (1-5) at the beginning of this notebook have been run successfully.
    *   Dependencies installed.
    *   API keys configured.
    *   Google Drive mounted.
    *   File paths defined (especially `DRIVE_PROJECT_BASE_PATH`).
    *   Paths to any necessary assets (Kokoro models, ComfyUI workflow, endscreen video) in Cell 5 are correct and the files exist in your Google Drive.
2.  **Prepare Input File**:
    *   In the `main()` function cell directly above this markdown cell, locate the `input_source_path` variable.
    *   Modify this path to point to **your** input file.
        *   For text-based input (e.g., an article or script you wrote), place your file (e.g., `my_script.txt` or `my_article.json`) inside the `input/text_sources` directory in your `DRIVE_PROJECT_BASE_PATH`. Then update `input_source_path` to something like `str(INPUT_DIR / "text_sources" / "my_script.txt")`.
        *   For video URL input (e.g. to summarize a YouTube video), create a `.txt` file (e.g., `my_video_link.txt`) containing the URL on the first line. Place this file inside `input/link_sources` in your `DRIVE_PROJECT_BASE_PATH`. Then update `input_source_path` like `str(INPUT_DIR / "link_sources" / "my_video_link.txt")`.
    *   If using `yt-dlp` for private videos, you might need a cookies file. If so, upload `cookies.txt` (or similar) to your `ASSETS_DIR` and uncomment/set `os.environ['YTDLP_COOKIES_FILE'] = str(ASSETS_DIR / 'cookies.txt')` in a new code cell before the `main()` call, or modify the setup cell 5.
3.  **Run the Pipeline**: Execute the code cell below this markdown cell, which contains the `main()` call.

In [44]:
import shutil
import os
import pathlib

path_to_delete = pathlib.Path("/content/newvidlands")

if path_to_delete.exists():
    print(f"Deleting {path_to_delete}...")
    try:
        shutil.rmtree(str(path_to_delete))
        print(f"Successfully deleted {path_to_delete}")
    except OSError as e:
        print(f"Error deleting {path_to_delete}: {e}")
else:
    print(f"Path not found: {path_to_delete}. Nothing to delete.")

Deleting /content/newvidlands...
Successfully deleted /content/newvidlands
